## Anomaly Detection - Credit Card Fraud Analysis

* Introduction
   - Anomaly detection identifies unusual patterns that do not follows the expected behaviour, called outliers. In this notebook, we use anomaly detection to credit card fraud using simple statistics and machine learning (Isolation Forest, LOF) approaches.

* Problem statement
   - Using the same Kaggle dataset on Credit Card Fraud Detection, model past credit card transactions to identify anomalies that correspond to fraudulent transaction. Aim for higher fraudulent transactions while minimising false positive.

* Dataset
   - Data available at [Kaggle - Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)

### Import libraries

In [ ]:
# For EDA
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

# Anomaly detection models
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

# Evaluation metrics
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42

### Data loading

In [ ]:
# Load data from tmp folder - dataset has been downloaded here from CC Fraud Detection project
df = pd.read_csv("/tmp/creditcard.csv")
df.head()

In [ ]:
# Inspect data structure
df.shape
#df.info

In [ ]:
#df.describe()

In [ ]:
# Check missing values
df.isnull().sum()

### Check 'Class' distribution (0 = Valid, 1 = fraud)

In [ ]:
# Check class distribution to determine fraud and valid transactions in the dataset
# Get the count of each class
fraud_count = df["Class"].value_counts()
print(f"Valid transactions: {fraud_count[0]}")
print(f"Fraudulent transactions: {fraud_count[1]}")
print(f"Fraud percentage: {fraud_count[1] / df.shape[0] * 100:.3f}%")


sns.countplot(x="Class", data=df, palette="coolwarm")
plt.title("Class Distribution: Valid vs Fraud")
plt.xticks([0, 1], ["Valid", "Fraud"])
plt.xlabel("Class")
plt.ylabel("Frequency")
plt.show()



### Check transaction amount distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df[df['Class'] == 0]['Amount'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Valid Transaction Amounts')
axes[0].set_xlabel('Amount')
axes[0].set_ylabel('Frequency')

axes[1].hist(df[df['Class'] == 1]['Amount'], bins=50, color='crimson', edgecolor='white')
axes[1].set_title('Fraudulent Transaction Amounts')
axes[1].set_xlabel('Amount')

plt.tight_layout()
plt.show()

### Check transaction time distribution

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

sns.histplot(data = df[df['Class'] == 0], x = 'Time', bins = 250, color = 'steelblue', edgecolor = 'white', ax = axes[0])
axes[0].set_title('Valid Transactions Over Time')
axes[0].set_xlabel('Time (seconds)')
axes[0].set_ylabel('Frequency')

sns.histplot(data = df[df['Class'] == 1], x = 'Time', bins = 250, color = 'crimson', edgecolor = 'white', ax = axes[1] )
axes[1].set_title('Fraudulent Transactions Over Time')
axes[1].set_xlabel('Time (seconds)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

*Observation*
- Valid transactions show a clear cyclical pattern with two distinct peaks (~75,000s and ~150,000s) and two quieter periods across the dataset, suggesting activity follows a regular pattern over time.

- Fraudulent transactions show no distinguishing pattern, they are sparse and noisy across all time windows with occassional spikes but no particular trend.

- This suggests `Time` has low discriminatory power for fraud detection and can be dropped during preprocessing.


### Check transaction patterns by amount over time

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

sns.scatterplot(data = df[df['Class'] == 0], x = 'Time', y = 'Amount', color = 'steelblue', alpha = 0.5, s = 7, ax = axes[0])
axes[0].set_title('Valid Transactions Amount Over Time')
axes[0].set_xlabel('Time (seconds)')
axes[0].set_ylabel('Amount (USD $)')

sns.scatterplot(data = df[df['Class'] == 1], x = 'Time', y = 'Amount', color = 'crimson', alpha = 0.5, s = 10, ax = axes[1])
axes[1].set_title('Fraudulent Transactions Amount Over Time')
axes[1].set_xlabel('Time (seconds)')
axes[1].set_ylabel('Amount (USD $)')

plt.tight_layout()
plt.show()

*Observations*
- As expected, the scatterplot mirrored the histograms above, valid transactions clustered densely at low amounts across all time windows with a few outliers  exceeding USD 20K $.
- Fraudulent transactions are sparse across all time windows with no discernible temporal pattern. Fraud transaction amounts are noticebaly lower than valid transactions, with max just above USD 2K $.
- Interestingly, the outliers in high-value valid transactions can be flagged as fraud using amount-based thresholding alone. This suggest limitation in simple statistics methods.
- This highlights why ML-based anomaly detection across all dataset features is necessary to establish correlations, and distinguish fraud from valid high-value transactions. 

### Summary statistics

In [ ]:
print("Summary statistics by Class:")
df.groupby("Class")["Amount"].describe().round(2)

*Observation*
- Valid transactions have a mean amount of USD 88.29, while fraudulent transactions have a higher mean of USD 122.21.
- Despite the higher mean, fraud transaction amounts have a lower maximum (USD 2,125.87) compared to valid transactions (USD 25,691.16), suggesting fraudsters avoid very large transactions.
- Standard deviation is comparable between classes (valid: USD 250.11, fraud: USD 256.68), suggesting similar spread in transaction amounts overall. However, relative to their respective means, fraud transactions show greater variability — a std of USD 256.68 against a mean of USD 122.21 versus USD 250.11 against USD 88.29 for valid transactions.

### Features correlation

In [ ]:
correlation_matrix = df.corr()
fig = plt.figure(figsize = (12, 9))
sns.heatmap(correlation_matrix, cmap = 'coolwarm', vmin = -0.8, vmax = 0.8, square = True)
plt.title('Feature Correlation Matrix')
plt.show()

*Observation*
- The diagonal confirms each feature is perfectly correlated with itself (expected).
- V1-V28 features show very little correlation with each other, confirming they are independent PCA components by design.
- Several features show notable correlation with 'Class':
   - Positive correlation (orange/reddish in Class column): V2, V4, V8, V11 (increase correlation for fraudulent transactions)
   - Negative correlation (blue in Class column): V1, V3, V5, V7, V10, V12, V14, V16, V17, V18 (decrease correlation for fraudulent transactions)
- 'Amount' shows weak correlation with 'Class', reiterate earlier observation that transaction amount alone is insufficient to detect fraud.
- 'Time' shows almost no correlation with 'Class', further supporting our decision to drop it during data preprocessing for model building.
- The features V1-V28 are anonymised PCA components, so no correlation interpretation can be made on these features to real problem.

### Data preprocessing
Based on EDA findings:
- 'Time' is dropped — no distinguishing pattern observed between valid and fraudulent transactions.
- 'Amount' is scaled using StandardScaler as it is the only non-PCA feature retained.
- V1–V28 are already PCA-transformed and do not require preprocessing.


### Scale & Prepare Features

In [ ]:
# Drop Time feature
df = df.drop(columns=['Time'])

# Scale Amount
scaler = StandardScaler()
df['Amount'] = scaler.fit_transform(df[['Amount']])

# Separate features and target outcome
X = df.drop(columns=['Class'])
y = df['Class']

print(f"Features shape: {X.shape}")
print(f"Fraud cases: {y.sum()} ({y.sum() / len(y) * 100:.3f}%)")


### Split dataset into train/test ratio (80/20)
- Use stratify=y to control data split. This ensures the fraud/normal ratio is preserved in both train and test splits since dataset show severe class imbalance.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y)

print(f"Training data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}")
print(f"Fraud in test set: {y_test.sum()}")

### Model building for anomaly detection
- We will use two ML models to do anomaly detection on the dataset:
1) Isolation Forest model
   - Isolation Forest isolates anomalies by randomly selecting a feature and splitting on a random value between the feature's min and max. Based on anomalies are few and different, they require fewer splits to isolate than normal observations, resulting in shorter path lengths in the decision tree - enabling anomaly to be isolated.
   
   - An anomaly score is calculated based on the path length. The shorter the path, the more likely the observation is an anomaly.
   
2) Local Outlier Factor (LOF)
   - LOF is a density-based method that computes the local density deviation of each data point relative to its neighbours. Points in significantly lower density regions than their neighbours are flagged as outliers.
   
   - The parameter 'n_neighbors' controls how many neighbours are considered. Points can be local outliers relative to its cluster. A value of 20 is recommended as a general default.

### Isolation Forest model


### Local Outlier Factor (LOF)

